In [19]:
from dotenv import load_dotenv
import os

load_dotenv()

True

# 定义模型
需要多模态的模型

In [20]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="qwen3.6-plus",  # 多模态
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 定义web搜索工具

In [21]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results=5,
    topic="general" 
)

# 添加记忆管理

In [23]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 连接sqlite
connection = sqlite3.connect('resources/personal_chief.db', check_same_thread=False)
# 初始化
checkpoint = SqliteSaver(connection)
# 自动建表
checkpoint.setup()

# 定义智能体

In [24]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：

1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份 “当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以 “可用食材清单” 为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpoint
)

# 测试

In [25]:
from langchain.messages import HumanMessage

multimodal_message = HumanMessage(content=[
    {"type": "text", "text": "帮我看看能做什么？"},
    {"type": "image_url", "image_url": {"url": "https://img95.699pic.com/photo/50274/7184.jpg_wh860.jpg"}}
])

config = {"configurable": {"thread_id": "1"}}

In [27]:
# 阻塞调用
response = agent.invoke({"messages": [multimodal_message]}, config)

In [28]:
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么？'}, {'type': 'image_url', 'image_url': {'url': 'https://img95.699pic.com/photo/50274/7184.jpg_wh860.jpg'}}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_9043c8ae534248deb93b6614)
 Call ID: call_9043c8ae534248deb93b6614
  Args:
    query: 用冰箱里的食材（小番茄、紫甘蓝、西兰花、胡萝卜、生菜、奶酪、苹果、橙子、牛奶）可以做什么菜谱
    search_depth: basic
    topic: general
================================= Tool Message =================================
Name: tavily_search

{"query": "用冰箱里的食材（小番茄、紫甘蓝、西兰花、胡萝卜、生菜、奶酪、苹果、橙子、牛奶）可以做什么菜谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.reddit.com/r/Cooking/comments/cjj8qs/if_you_tell_me_whats_in_your_fridge_or_pantry_ill/?tl=zh-hans", "title": "如果你告诉我你冰箱或储藏室里有什么，我就给你写个简单的菜谱", "content": "如果你告诉我你冰箱或储藏室里有什么，我就给你写个简单的菜谱，把你的食材用光光！", "score": 0.64639384, "raw_content

In [29]:
# 记忆能力测试
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第三道菜，可以说的更详细一点吗？")]},
    config
)

In [30]:
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么？'}, {'type': 'image_url', 'image_url': {'url': 'https://img95.699pic.com/photo/50274/7184.jpg_wh860.jpg'}}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_9043c8ae534248deb93b6614)
 Call ID: call_9043c8ae534248deb93b6614
  Args:
    query: 用冰箱里的食材（小番茄、紫甘蓝、西兰花、胡萝卜、生菜、奶酪、苹果、橙子、牛奶）可以做什么菜谱
    search_depth: basic
    topic: general
================================= Tool Message =================================
Name: tavily_search

{"query": "用冰箱里的食材（小番茄、紫甘蓝、西兰花、胡萝卜、生菜、奶酪、苹果、橙子、牛奶）可以做什么菜谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.reddit.com/r/Cooking/comments/cjj8qs/if_you_tell_me_whats_in_your_fridge_or_pantry_ill/?tl=zh-hans", "title": "如果你告诉我你冰箱或储藏室里有什么，我就给你写个简单的菜谱", "content": "如果你告诉我你冰箱或储藏室里有什么，我就给你写个简单的菜谱，把你的食材用光光！", "score": 0.64639384, "raw_content

In [31]:
# 只截取需要的信息
response["messages"][-1].pretty_print()

================================== Ai Message ==================================

好的，既然您对**混合蔬菜沙拉（生菜 + 紫甘蓝 + 小番茄 + 奶酪）**感兴趣，我来为您详细拆解这道菜的制作过程。这道菜虽然是“凉菜”，但只要处理得当，口感会非常爽脆开胃，而且奶酪的加入会让味道更醇厚。

以下是详细的**厨师级操作指南**：

### 🥗 菜名：缤纷时蔬奶酪沙拉

#### 📋 食材准备（基于您冰箱里的库存）
*   **主料**：
    *   **生菜**：约 3-4 片（洗净沥干）
    *   **紫甘蓝**：约 1/4 个（洗净）
    *   **小番茄**：8-10 颗（洗净）
    *   **奶酪**：约 30g（看您冰箱里那块黄色的奶酪，如果是切达/高达类硬质奶酪，建议切丁或擦丝；如果是菲达/Feta 类软奶酪，直接捏碎即可）
*   **灵魂油醋汁**（家中常备即可）：
    *   橄榄油 2 勺
    *   苹果醋（或柠檬汁/白醋）1 勺
    *   盐 少许（约 2 克）
    *   黑胡椒碎 少许（可选）

---

#### 🔪 详细制作步骤

**第一步：处理食材（关键在刀工与水分）**
1.  **生菜**：洗净后，**一定要用厨房纸吸干水分**。水分太多会稀释沙拉汁，导致味道变淡。用手撕成一口大小的块（手撕比刀切更不容易氧化变色）。
2.  **紫甘蓝**：
    *   切成极细的丝（像头发丝一样细口感最好）。
    *   **厨师秘诀**：紫甘蓝口感较硬，切好后放入碗中，撒一点点盐，用手抓捏腌制 3 分钟，这样它能变软，吃起来更脆嫩。
3.  **小番茄**：对半切开（如果是很小的樱桃番茄，保持完整也可以）。

**第二步：调制油醋汁（万能公式）**
*   找一个小碗，按照 **3:1 的比例**倒入橄榄油和醋（例如 2 勺油配 1 勺醋）。
*   加入少许盐和黑胡椒。
*   用叉子快速搅拌，直到油和醋乳化（看起来变得稍微浓稠），这样酱汁才能挂在蔬菜上。

**第三步：混合与装盘**
1.  找一个大碗，先放入处理好的生菜和腌软的紫甘蓝。
2.  放入切开的小番茄。
3.  倒入调好的油醋汁，用筷子或夹子轻轻

# 流式调用

In [ ]:
from langchain.messages import AIMessageChunk

for chunk, metadata in agent.stream(
    {"messages": [multimodal_message]},
    config,
    stream_mode="messages"
):
    if isinstance(chunk, AIMessageChunk) and chunk.content:
        print(chunk.content, end="", flush=True)

# 获取会话历史

In [ ]:
checkpoint.get(config)

In [ ]:
for m in checkpoint.get(config)['channel_values']['messages']:
    print(m)

定义便捷获取会话信息的函数

In [ ]:
from langchain.messages import AIMessage

def get_messages(thread_id: str) -> list[dict[str, str]]:
    # 根据thread_id查询checkpoint
    cp = checkpoint.get({"configurable": {"thread_id": thread_id}})

    # 不存在返回空列表
    if not cp:
        return []
    
    # 安全获取messages
    channel_values = cp.get('channel_values')
    if not channel_values:
        return []
    
    messages = channel_values.get('messages', [])
    if not messages:
        return []
    
    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue

        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})

    return result

In [ ]:
get_messages("1")

# 清空会话历史

In [ ]:
checkpoint.delete_thread('1')